# UR5e 焊接机械臂摆动施焊 — PyVista 交互式演示

本 notebook 把两条线索合成一个**可交互的三维场景**:

- **机械臂**: `SixDofArm` (模块 7b) 以 **Universal Robots UR5e** 参数化
  (`conf/model/robot6_ur5e.yaml`)。选型: UR 系列中按臂展比对 —
  UR3e (500 mm) / **UR5e (850 mm)** / UR10e (1300 mm), 焊缝布置在基座前
  450 mm 处, UR5e 的 850 mm reach 与原默认球腕臂一致, 工作空间中心区
  不变。DH 表与连杆质量取自 UR 官方运动学/动力学参数表 (标准 DH,
  **偏置腕** — 腕轴不交于一点, §2 的三段腕清晰可见); `r_link` /
  `J_rotor` 仍为本模型近似 (圆柱惯量 + 折算转子惯量), 非 UR 数据。
  连杆按实心圆柱建模, 渲染直接用同一几何 (`_kin(q)` 帧原点)。
- **摆动施焊**: 《机器人部署场景》(`robot_deployment_scenarios.ipynb`) 的
  流程 — 数据库中位工况 + 2 Hz × 4 mm 三角摆, `track_path` 强迫 DEL
  跟踪, 实际 TCP 轨迹经 `RobotExecutedWeave` 注入 `GoldakFDM`
  (§4 超细网格 `xfine`, 分辨摆动节距)。§4b 另解一遍**对流增强**温度场
  (模块 10A 有效导热率耦合, `convection=effective`), §5 可用复选框
  叠加对比; §5b 输出焊缝成形动画 GIF (`results/robot6_weave_seam.gif`)。

所有场景都由 PyVista 的 **server widget** 持有同一个 VTK render window：
可播放/拖动时间轴、原位切换图层而不丢相机视角。普通拖动控制相机；
点击对象出现黄框，**Shift+左拖**平移机械臂或工艺对象组，
**Ctrl+左拖**绕组中心旋转。§5/§5b 与 GIF 最终合并为三页 tab。

## 前置条件

安装交互式 notebook 依赖 (含 pyvista + jupyter):

```bash
uv sync --extra notebook
uv run jupyter lab      # 或 uv run jupyter notebook
```

全程约 3–4 分钟 (跟踪仿真 ~1 min + xfine 传导场 ~30 s + fine 对流场
~20 s + §5b GIF ~1 min)。工艺/摆动的定量部署分析 (摆频扫描、幅值保真度)
见 `robot_deployment_scenarios.ipynb`。

## 0. 选择内联后端

- `'server'` — 浏览器事件回传到持久 VTK render window；逐帧等值面、
  图层切换和对象选择/变换都需要此模式（本 notebook 的默认值）。
- `'static'` — 仅作为无显示环境的截图回退；不支持播放或对象操作。

远程 Jupyter 若通过反向代理访问，可按部署设置 Trame server proxy。
VS Code 受限 webview 通常无法连接 live server，推荐浏览器中的 JupyterLab。

In [ ]:
import pyvista as pv
from welding_dynamics import ensure_display

# 无头 / 远程内核里继承的 $DISPLAY 可能已失效 (如 :1 已死)。VTK 需要活跃的 GL
# 上下文, 否则 vtkXOpenGLRenderWindow 会直接 Abort 杀掉内核。ensure_display()
# 会探测显示, 必要时自动拉起 Xvfb 虚拟帧缓冲 (macOS/Windows 走原生 GL, 无需 X)。
ensure_display()

# 强制 server rendering：鼠标 pick/对象变换与动态 contour 都发生在
# Python 侧同一个 VTK render window，不能在 client/html 模式中降级。
pv.set_jupyter_backend('server')

## 1. 跟踪仿真: 数据库中位工况 + 2 Hz × 4 mm 三角摆

与《机器人部署场景》同一设置: 焊缝沿机器人基座 x 轴 (工作空间中心区,
平焊, 枪尖竖直向下), 参考轨迹 = 恒速焊缝 + 摆动指令, `track_path`
强迫 DEL 全动力学跟踪 5 s。UR 偏置腕没有球腕解耦, 位姿逆解仍走同一
DLS 数值 IK — 唯一要额外指定的是**解支**: `Q_SEED` 选肘上抬、基座朝前
的典型 UR 作业姿态 (随机种子会落到卷绕/肘下解支)。另按相同参考做一遍
**运动学 IK 采样** (251 帧), 供后面渲染机械臂姿态 — 实际关节状态与之
只差亚毫米, 在场景尺度上不可分辨。

In [ ]:
import numpy as np
from hydra import compose, initialize_config_module
from hydra.utils import instantiate

from welding_dynamics.config import arc_power   # 导入即注册 wd.* 解析器
from welding_dynamics import RobotExecutedWeave


def compose_cfg(config_name, *overrides):
    with initialize_config_module(config_module="welding_dynamics.conf",
                                  version_base="1.3"):
        return compose(config_name=config_name, overrides=list(overrides))


# UR5e 参数化 (model@robot6=robot6_ur5e); CLI 等价 welding-sim-vi model@robot6=robot6_ur5e
# solver=xfine (0.5 mm): 摆动节距 v/f ≈ 2.6 mm 需 ~5 格才不出夸大的"鱼鳞"棱片
# (fine 的 0.8 mm 下仅 ~3 格, 接近网格 Nyquist, 棱脊间距被网格拍频调制成 1.6–4 mm)
arm = instantiate(compose_cfg("sim_vi", "model@robot6=robot6_ur5e").robot6)
cfg = compose_cfg("sim_3d", "process=db_median", "solver=xfine")
weave = instantiate(compose_cfg("sim_3d", "weave=triangle").weave)
V_WELD = float(cfg.process.travel_speed_m_s)          # 5.15 mm/s
Q_ARC = arc_power(cfg)                                # 8120 W (U·I)
P0 = np.array([0.45, 0.0, 0.25])                      # m 焊缝起点 (板顶面)
T_TRACK = float(cfg.solver.t_end)                     # 5 s
R_REF = np.diag([1.0, -1.0, -1.0])                    # 平焊: 枪尖竖直向下
Q_SEED = (0.3, -2.0, -1.6, 2.1, -1.57, 1.9)           # UR 肘上抬/基座朝前解支

print(f"UR5e: 连杆质量 {[float(x) for x in arm.m]} kg, reach 850 mm, "
      f"|a2|+|a3| = {(abs(arm.dh_a[1]) + abs(arm.dh_a[2]))*1e3:.0f} mm, "
      f"偏置腕 d5 = {arm.dh_d[4]*1e3:.1f} mm")


def p_ref(t):
    dx, dy = weave.offset(t)
    return P0 + np.array([V_WELD*t + dx, dy, 0.0])


t_tr, tip, ref, err = arm.track_path(p_ref, T_TRACK, q_seed=Q_SEED)
print(f"跟踪 {weave.describe()}: 三维 RMS {1e3*np.sqrt((err**2).mean()):.2f} mm, "
      f"执行横向峰-峰 {1e3*np.ptp(tip[:, 1]):.2f} mm (指令 4.00 mm)")

# 渲染用姿态帧: 沿参考轨迹的 IK 采样 (热启动保持解支连续)
DTQ = T_TRACK/250
q_grid, q_seed = [], Q_SEED
for tk in np.arange(251)*DTQ:
    q_seed = arm.ik(p_ref(tk), R_REF, q0=q_seed)
    q_grid.append(q_seed)
q_grid = np.array(q_grid)


def q_at(t):
    return q_grid[min(250, max(0, int(round(t/DTQ))))]


print(f"姿态帧: {len(q_grid)} 帧 @ {DTQ*1e3:.0f} ms")

## 2. 机械臂场景 (焊接中途 t = 2.5 s)

连杆圆柱 + 关节球 + 焊枪锥都由 `arm._kin(q)` 的帧原点直接生成 —
渲染几何与动力学模型是同一份 UR5e DH 表, **偏置腕的三段腕链**
(wrist-1/2/3 互不相交的转轴) 在枪头附近清晰可见。红色细管为
**实际执行**的 TCP 轨迹 (含摆动), 黑线为焊缝中心线。

**取景有两套互补方式**:

- **相机鼠标**: 左拖旋转，中拖平移，右拖/滚轮缩放。
- **对象鼠标**: 点击机械臂或工艺对象按逻辑组选择（黄框）；
  **Shift+左拖**平移所选组，**Ctrl+左拖**绕组中心旋转。对象拖动
  发生在 server widget 的同一 VTK 场景中，不重建 iframe，也不丢相机。
- **视图滑块** (两行: 平移 x/y/z [mm] + 方位/俯仰/滚转 [°] +
  **臂透明 [%]**): 服务端参数, 精确、可复现, 且**自动持久化** —
  每次改动写入 `.robot6_view_state.json` (gitignored), 重跑 notebook /
  重启内核后自动恢复上次的取景 (§3 的时间滑块与 §5 的复选框同样持久化)。

**臂透明滑块**渐隐机械臂以透视焊缝/TCP 轨迹; 拉到 **100% 时整臂移出
场景** (半透明 actor 仍参与深度混合, 只有不画才保证零遮挡)。

**瞬时熔池** (`g.T ≥ Tm`, 当下的池, 而非 `g.peak` 的熔合区历史; 求解器
只保留末时刻场, 故它是 t = 5 s 焊末的池, 固定在焊缝末端)。
图层复选框在持久 VTK pipeline 内切换 `g.T` / `g.peak`，相机和对象
位姿都保持不变；机械臂透明度、图层状态和可复现取景继续持久化。

需先跑 §4 才有热场；§4 完成后会把热场原位接入已显示的 §2/§3 场景。
本格建立的 `RobotWeaveContext` 和 `PyVistaWidgetApp` 供后续场景复用。

In [ ]:
import json
import sys
from pathlib import Path

import ipywidgets as widgets
from IPython.display import HTML, display

from utils import (CompositePVScene, LayerSpec, PyVistaWidgetApp,
                   RobotPVScene, RobotWeaveContext, SeamPVScene,
                   WidgetStore, build_output_tabs, close_app,
                   close_output_tabs, initial_frame)

R_LINK = [45.0, 38.0, 32.0, 22.0, 20.0, 16.0]    # mm 连杆显示半径 (UR5e 目测锥度)
R_JOINT = [50.0, 42.0, 36.0, 26.0, 24.0]         # mm 关节球


def add_arm(p, q, opacity=1.0):
    """按 _kin(q) 画机械臂 (单位 mm): 连杆圆柱 / 关节球 / 焊枪锥 / 底座。

    ``opacity`` < 1 时整臂半透明 (焊缝/TCP 轨迹/熔池透视可见);
    <= 0.02 时整臂直接不加入场景 —— "100% 透明"保证零遮挡
    (半透明 actor 仍参与深度混合, 只有移出场景才完全不挡)。
    """
    if opacity <= 0.02:
        return
    o, z, R = arm._kin(q)
    o = o*1e3
    tz = R[:, 2]                                  # 焊枪轴向 (工具 z, 指向工件)
    ends = o.copy()
    ends[6] = o[6] - tz*50.0                      # 连杆 6 画到锥根, 枪尖由锥体表现
    for i in range(1, 7):
        seg = ends[i] - o[i-1]
        L = np.linalg.norm(seg)
        if L > 1e-3:                              # 球腕处 o4==o5, 跳过零长连杆
            p.add_mesh(pv.Cylinder(center=0.5*(o[i-1] + ends[i]), direction=seg,
                                   radius=R_LINK[i-1], height=L),
                       color="#a7a9ac", smooth_shading=True, opacity=opacity)
    for i in range(1, 6):
        p.add_mesh(pv.Sphere(radius=R_JOINT[i-1], center=o[i]),
                   color="#57a7c6", smooth_shading=True, opacity=opacity)
    p.add_mesh(pv.Cone(center=o[6] - tz*27.5, direction=tz,
                       height=55.0, radius=11.0),
               color="#c0392b", smooth_shading=True, opacity=opacity)
    p.add_mesh(pv.Cylinder(center=(0, 0, -15.0), direction=(0, 0, 1),
                           radius=90.0, height=30.0), color="#6b7078",
               opacity=opacity)


def add_workpiece_and_traces(p, t_now=None):
    """工件板 + 焊缝中心线 + 实际 TCP 轨迹 (可截断到 t_now)。"""
    p.add_mesh(pv.Box(bounds=(P0[0]*1e3 - 90, P0[0]*1e3 + 110, -70, 70,
                              P0[2]*1e3 - 20, P0[2]*1e3)),
               color="#d8d2c4", opacity=0.55)
    seam = np.array([P0*1e3, (P0 + [V_WELD*T_TRACK, 0, 0])*1e3]) + [0, 0, 0.3]
    p.add_mesh(pv.lines_from_points(seam), color="black", line_width=3)
    m = slice(None) if t_now is None else t_tr <= t_now
    pts = tip[m]*1e3 + [0, 0, 0.3]                # 抬 0.3 mm 避免与板面 z-fight
    if len(pts) > 2:
        p.add_mesh(pv.lines_from_points(pts).tube(radius=0.6), color="#d81b1b")


# ---- 瞬时熔池/熔合区历史 双层: 场景内复选框**客户端**切换, 视角不变 ----
# 自包含 html 场景无回传通道, 服务端重绘必然重置鼠标视角。因此把两组
# actor **都**导出, 用透明度指纹标记 (万分位 1 = 瞬时组, 2 = 峰值组,
# 视觉不可分辨; index.json 保留全精度已验证; 隐藏 actor 不会被导出,
# 故两组都可见导出, 加载后由注入脚本立即按初值设可见性)。切换通过
# vtk.js 加载器暴露的 window.global.renderWindow 翻转可见性并重渲染
# —— 相机完全不动。
TAG_INST, TAG_PEAK = 1e-4, 2e-4                   # 加到基准透明度上的指纹


def add_instant_pool(p):
    """加入瞬时熔池 (g.T ≥ Tm 等值面, 瞬时组指纹) — 当下的池, 而非
    g.peak 的熔合区历史。求解器只保留末时刻场, 故这是 t = 5 s (焊末)
    的池, 固定在焊缝末端 (§3 把 t 拖到 5 s 时恰与枪尖对齐)。
    返回是否成功 (§4 未运行时 g 不存在 -> False)。"""
    if 'g' not in globals():
        return False
    ggx = (g.x - cfg.run.goldak.x_start + P0[0])*1e3
    ggy = (g.y + P0[1])*1e3
    ggz = (P0[2] - g.z)*1e3
    X, Y, Z = np.meshgrid(ggx, ggy, ggz, indexing='ij')
    sg = pv.StructuredGrid(X, Y, Z)
    sg['T'] = g.T.ravel(order='F')
    p.add_mesh(sg.contour([float(g.Tm)], scalars='T'),
               color=(0.85, 0.1, 0.1), opacity=0.9 + TAG_INST)
    return True


# 注入的场景内切换脚本: 只用单引号 (srcdoc 属性里双引号已转义为 &quot;)
_TOGGLE_JS = """
(function(){
  var tries = 0;
  function grp(a){var op=a.getProperty().getOpacity();
    var f=Math.round(op*10000)%10; return f===1?1:(f===2?2:0);}
  function apply(showInst){
    var g=window.global; if(!g||!g.renderWindow) return false;
    var rr=g.renderWindow.getRenderers(); var found=false;
    for(var j=0;j<rr.length;j++){          // 场景 actor 在同步器新建的
      var acts=rr[j].getActors();          // renderer 里 (不是 rr[0]), 全扫
      for(var i=0;i<acts.length;i++){var k=grp(acts[i]);
        if(k===1){acts[i].setVisibility(showInst);found=true;}
        if(k===2){acts[i].setVisibility(!showInst);found=true;}}}
    if(found) g.renderWindow.render();
    return found;
  }
  function init(){
    if(!apply(__INIT__)){ if(tries++ < 40) setTimeout(init,150); return; }
    var d=document.createElement('div');
    d.style.cssText='position:absolute;top:6px;left:50%;'+
      'transform:translateX(-50%);z-index:10;background:rgba(255,255,255,.88);'+
      'padding:3px 8px;border-radius:4px;font:12px sans-serif;color:#333;';
    var c=document.createElement('input'); c.type='checkbox';
    c.checked=__INIT__; c.onchange=function(){apply(c.checked);};
    var l=document.createElement('label'); l.style.cursor='pointer';
    l.appendChild(c); l.appendChild(document.createTextNode(' __LABEL__'));
    d.appendChild(l); document.body.appendChild(d);
  }
  setTimeout(init,150);
})();"""

_ANCHOR = "OfflineLocalView.load(container, { base64Str });"


def inject_pool_toggle(html, label, initial):
    """向导出 html (iframe srcdoc) 注入场景内切换复选框脚本。"""
    js = (_TOGGLE_JS.replace('__INIT__', 'true' if initial else 'false')
                    .replace('__LABEL__', label))
    assert html.count(_ANCHOR) == 1, "html 结构变化: 未找到注入锚点"
    return html.replace(_ANCHOR, _ANCHOR + js)


# ---- 视图控件: 平移/旋转/臂透明度滑块, 值持久化 JSON, 重跑自动复原 ----
VIEW_STATE = Path(".robot6_view_state.json")      # 相对 notebooks/, 已 gitignore
try:
    _view_state = json.loads(VIEW_STATE.read_text())
except Exception:
    _view_state = {}


def tracked(scene, name, w):
    """恢复 (scene, name) 上次存储的值, 并在每次变化时写回 JSON。

    先恢复再挂 observer, 恢复本身不落盘; 之后任何控件变化 (滑块、复选框,
    含 §3 的 t 滑块) 都同步写 .robot6_view_state.json —— 重跑 notebook
    时 `tracked` 会把控件初始化回上次的值, interactive_output 的首帧
    就按存储的取景渲染。
    """
    val = _view_state.get(scene, {}).get(name)
    if val is not None:
        w.value = val

    def _save(change):
        _view_state.setdefault(scene, {})[name] = change['new']
        VIEW_STATE.write_text(json.dumps(_view_state, indent=1,
                                         ensure_ascii=False))

    w.observe(_save, 'value')
    return w


POOL_LABEL = '瞬时熔池 g.T (当下的池; 不勾 = 熔合区历史 g.peak)'


def pool_checkbox(scene, extra=''):
    """瞬时熔池**初值**复选框 (持久化, 改动触发重绘)。即时切换请用场景
    顶部注入的复选框 —— 客户端翻转可见性, **视角保持不变**。"""
    return tracked(scene, 'inst', widgets.Checkbox(
        value=False, indent=False, layout=widgets.Layout(width='760px'),
        description=f'{POOL_LABEL} — 初值/持久化{extra}'))


VIEW_KEYS = ('px', 'py', 'pz', 'az', 'el', 'roll', 'arm_t')


def view_widgets(scene, lim=300.0, step=10.0):
    """七个视图滑块 (顺序同 VIEW_KEYS): 平移 x/y/z [mm], 方位/俯仰/滚转 [°],
    臂透明 [%] (0 = 不透明, 100 = 整臂移出场景, 完全不遮挡焊缝/熔池)。
    松开才重绘 (~0.5 s); 全部经 `tracked` 持久化。"""
    kw = dict(continuous_update=False, readout_format='.0f',
              layout=widgets.Layout(width='230px'))
    pans = [tracked(scene, f'p{ax}',
                    widgets.FloatSlider(min=-lim, max=lim, step=step, value=0.0,
                                        description=f'平移 {ax} [mm]', **kw))
            for ax in 'xyz']
    rots = [tracked(scene, key,
                    widgets.FloatSlider(min=mn, max=mx, step=2.0, value=0.0,
                                        description=f'{lab} [°]', **kw))
            for key, lab, mn, mx in (('az', '方位角', -180.0, 180.0),
                                     ('el', '俯仰角', -80.0, 80.0),
                                     ('roll', '滚转', -180.0, 180.0))]
    alpha = tracked(scene, 'arm_t',
                    widgets.FloatSlider(min=0.0, max=100.0, step=5.0, value=0.0,
                                        description='臂透明 [%]', **kw))
    return pans + rots + [alpha]


def apply_view(p, px, py, pz, az, el, roll):
    """在场景默认取景基础上施加视图参数 (绝对量, 每次重绘从默认相机起算,
    故取景可复现): 先世界坐标平移 (相机+焦点同移, 视线方向不变), 再绕
    焦点做方位角/俯仰角/滚转 (vtkCamera.Azimuth/Elevation/Roll; 每步
    OrthogonalizeViewUp 防上方向漂移, 俯仰限 ±80° 避开极点翻转)。"""
    pan = np.array([px, py, pz], dtype=float)
    cam = p.camera
    cam.focal_point = tuple(np.asarray(cam.focal_point) + pan)
    cam.position = tuple(np.asarray(cam.position) + pan)
    if az:
        cam.Azimuth(az)
        cam.OrthogonalizeViewUp()
    if el:
        cam.Elevation(el)
        cam.OrthogonalizeViewUp()
    if roll:
        cam.Roll(roll)


def add_mouse_hint(p):
    """鼠标操作角标。html 场景的 vtk.js trackball 交互全挂在左键+修饰键:
    左键拖拽=旋转, Shift+左键=平移(即时, 无需重绘), Ctrl/Alt+左键=自旋,
    滚轮=缩放。VTK 默认字体无中文字形, 提示用英文。注意: 自包含 html
    场景没有回传通道 (这正是它在 VS Code webview 里能用的原因), 鼠标
    取景只活在浏览器端 —— 滑块才是可持久化/可复现的取景, 二者互补。"""
    p.add_text("drag: rotate | Shift+drag: pan | scroll: zoom",
               position='upper_right', font_size=8, color="#888888")


_STD_STREAMS = (sys.stdout, sys.stderr)           # 本格进入时的原始内核流


def html_view(p, pool_toggle=None):
    """导出自包含 vtk.js 场景 ('html' 后端, 无 trame 服务器, 取 iframe 以
    text/html 内联, VS Code / JupyterLab 均可渲染) 并**复原内核 std 流**:
    Linux 下 add_text 触发 VTK 的 matplotlib-mathtext 探测
    (vtkPythonInterpreter), 后者把 sys.stdout/stderr 换成**只读**的
    vtkPythonStdStreamCaptureHelper; IPython>=9 每格 run_cell 的 _tee 要对
    stream.write 赋值, 遇只读对象抛 AttributeError, 内核 execute_request
    中断且不回复 —— 表现为**下一格永远"运行中"**。渲染后换回原对象即可。

    ``pool_toggle=(label, initial)``: 注入场景内复选框, 客户端切换
    指纹标记的两组 actor (瞬时熔池/熔合区历史), **视角保持不变**。
    """
    try:
        viewer = p.show(jupyter_backend='html', return_viewer=True)
    finally:
        for name, orig in zip(("stdout", "stderr"), _STD_STREAMS):
            if type(getattr(sys, name)).__name__ == "vtkPythonStdStreamCaptureHelper":
                setattr(sys, name, orig)
    if not getattr(viewer, 'value', None):
        return viewer
    body = viewer.value
    if pool_toggle is not None:
        body = inject_pool_toggle(body, *pool_toggle)
    # <div> 包裹自包含 iframe: 否则 IPython>=? 的 HTML 把裸 <iframe>…</iframe>
    # 误判为应改用 IFrame, 每次渲染刷一条 UserWarning (display.py:444)。
    return HTML(f'<div>{body}</div>')


def show_arm_scene(px, py, pz, az, el, roll, arm_t, inst):
    p = pv.Plotter(notebook=True, window_size=(950, 620))
    p.set_background("white")
    add_arm(p, q_at(2.5), opacity=1.0 - arm_t/100.0)
    add_workpiece_and_traces(p, t_now=2.5)
    has_pool = add_instant_pool(p)
    if inst and not has_pool:
        p.add_text("instantaneous pool: run section 4 first",
                   position='lower_right', font_size=9, color="#b03030")
    p.add_text("t = 2.50 s", font_size=11, color="black")
    add_mouse_hint(p)
    p.add_axes(xlabel="x [mm]", ylabel="y [mm]", zlabel="z [mm]")
    p.view_isometric()
    apply_view(p, px, py, pz, az, el, roll)
    return html_view(p, pool_toggle=(POOL_LABEL, inst) if has_pool else None)


# ---- PyVista server widget：持久场景，不再为每次控件变化重建 html iframe ----
close_app(globals(), 'arm_pv')
pv_store = WidgetStore(Path('.robot6_view_state.json'))
pv_context = RobotWeaveContext(arm, q_at, T_TRACK, t_tr, tip, P0, V_WELD)
robot_layers = (
    LayerSpec('robot', 'UR5e robot'),
    LayerSpec('workpiece', 'Workpiece'),
    LayerSpec('seam', 'Seam centreline'),
    LayerSpec('executed_path', 'Executed TCP path'),
    LayerSpec('thermal', 'Thermal field after §4'),
    LayerSpec('instant_mode', 'Instantaneous field (off: peak history)', False),
    LayerSpec('axes', 'World axes'),
)
arm_scene = RobotPVScene(
    pv_context, [2.5], layers=robot_layers, grow_trace=True,
    pool_model=globals().get('g'),
    pool_x_start=float(cfg.run.goldak.x_start),
)
arm_pv = PyVistaWidgetApp(
    arm_scene, layers=robot_layers, store=pv_store, scene_name='arm',
    title='UR5e welding pose — PyVista live widget', height=660,
)
display(arm_pv.panel)

## 3. 播放/时间滑块: 沿摆动轨迹移动机械臂

Play 与帧滑块直接更新持久 robot/trace actor；红色轨迹随时间增长，
可清楚看到腕部随三角摆左右摆动。播放、切图层、透明度和对象变换
都不重建 render window，因此保留当前鼠标视角。

相机/对象手势同 §2；视图、图层和时间值持久化在
`.robot6_view_state.json`，重跑后从上次位置继续。§4 跑完后末时刻
瞬时场和峰值历史会原位接入本场景。

In [ ]:
def show_pose(t_val, px, py, pz, az, el, roll, arm_t, inst):
    p = pv.Plotter(notebook=True, window_size=(820, 540))
    p.set_background("white")
    add_arm(p, q_at(t_val), opacity=1.0 - arm_t/100.0)
    add_workpiece_and_traces(p, t_now=t_val)
    has_pool = add_instant_pool(p)
    if inst and not has_pool:
        p.add_text("instantaneous pool: run section 4 first",
                   position='lower_right', font_size=9, color="#b03030")
    p.add_text(f"t = {t_val:.2f} s", font_size=11, color="black")
    add_mouse_hint(p)
    p.add_axes(xlabel="x [mm]", ylabel="y [mm]", zlabel="z [mm]")
    p.view_isometric()
    apply_view(p, px, py, pz, az, el, roll)
    return html_view(p, pool_toggle=(POOL_LABEL, inst) if has_pool else None)


close_app(globals(), 'pose_pv')
pose_times = np.linspace(0.0, T_TRACK, 41)
pose_scene = RobotPVScene(
    pv_context, pose_times, layers=robot_layers,
    current_frame=initial_frame(pv_store, 'pose', pose_times, 2.5),
    grow_trace=True, pool_model=globals().get('g'),
    pool_x_start=float(cfg.run.goldak.x_start), size=(820, 540),
)
pose_pv = PyVistaWidgetApp(
    pose_scene, layers=robot_layers, store=pv_store, scene_name='pose',
    title='UR5e weave playback — PyVista live widget', height=590,
)
display(pose_pv.panel)

## 4. 机器人执行轨迹 → 熔池

把 §1 的实际 TCP 轨迹经 `RobotExecutedWeave` 注入 `GoldakFDM`
(数据库中位工况, **超细网格** `solver=xfine`; `amplitude_m > 0`
自动切换全宽网格), 求解 5 s 瞬态温度场。本格约 30 s。

为什么要 0.5 mm 网格: 峰值场熔合面的"鱼鳞"棱脊节距是物理的
(每半摆动周期一道, v/2f ≈ 1.3 mm), 但 fine (0.8 mm) 下一个摆动
周期只有 ~3 格 — 接近网格 Nyquist, marching-cubes 把棱脊渲染成
夸大的分离"硬币"棱片, 且间距被网格拍频调制 (1.6–4 mm 不等)。
xfine 下节距 ~5 格, 棱脊变成平滑扇贝纹, 间距回到物理值。

In [ ]:
rw = RobotExecutedWeave.from_tracking(t_tr, tip, P0, V_WELD,
                                      frequency_Hz=weave.frequency_Hz)
g = instantiate(cfg.goldak, Q=Q_ARC, weave=rw)
g.run(t_end=cfg.run.goldak.t_end, x_start=cfg.run.goldak.x_start)
L, W, D = g.pool_size()
print(f"{rw.describe()}")
print(f"网格 {g.Nx}x{g.Ny}x{g.Nz} (全宽), 熔池 L×W×D = {L:.1f}×{W:.1f}×{D:.1f} mm, "
      f"T_max = {g.peak.max():.0f} K")

# 已显示的 §2/§3 live scene 原位接入热场，不重建 widget/相机。
for live_scene in (globals().get('arm_scene'), globals().get('pose_scene')):
    if live_scene is not None and not live_scene.closed:
        live_scene.attach_pool(g, x_start=float(cfg.run.goldak.x_start))

## 4b. 熔池对流修正 (模块 10A 耦合)

同一执行轨迹再解一遍**对流增强**温度场 (`convection=effective`,
方案 A, 见 `docs/melt_convection_assessment.md`): 池内导热率放大
`α_eff/α` 倍 (10A 由 `dγ/dT` 折算, 限幅 6), 把 Marangoni 搅拌近似为
增强热输运。dt 随最大扩散率缩小 (×6 步数), 本格约 20–30 s。
预期: 峰值温度从非物理的 ~5700 K (超沸点) 回落到沸点以下, 池内混合
把"鱼鳞"棱脊抹平约 5 倍 (顶面熔线 mean|ΔT| 182 → 38 K)。

对流场特意留在 `solver=fine` (0.8 mm): 池内混合本来就把棱脊抹平了,
xfine 对它没有视觉收益, 却要 ~5 分钟 (×12 网格代价再 ×6 步数)。
§4 的 `g` 仍保留 xfine 终态精度；本格另以 `solver=fine` 对传导/对流
各做一次 0–5 s 求解并同时抓取 41 个 float32 快照。两组对齐快照由
§5 动态合成场和 §5b GIF/焊缝回放共用，不再重复求解。

In [ ]:
# 重跑本格前释放引用旧快照的下游 live scenes/tab。
close_output_tabs(globals())
close_app(globals(), 'composite_pv')
close_app(globals(), 'seam_pv')

# 40 个 125 ms 区间 + t=0 环境温度首帧；§5/§5b 共用。
N_FRAMES = 41
weld_frame_times = np.linspace(0.0, T_TRACK, N_FRAMES)
cfg_gif = compose_cfg("sim_3d", "process=db_median", "solver=fine")
x_start = float(cfg_gif.run.goldak.x_start)
g_gif = instantiate(cfg_gif.goldak, Q=Q_ARC, weave=rw)
snaps = g_gif.run_with_snapshots(
    t_end=T_TRACK, x_start=x_start, frame_times=weld_frame_times,
    snapshot_dtype=np.float32,
)

# convection=effective: Goldak 节点嵌套实例化 10A 对象。
cfg_conv = compose_cfg("sim_3d", "process=db_median", "solver=fine",
                       "convection=effective")
g_conv = instantiate(cfg_conv.goldak, Q=Q_ARC, weave=rw)
conv_x_start = float(cfg_conv.run.goldak.x_start)
if not np.isclose(x_start, conv_x_start):
    raise ValueError('传导/对流动画必须使用相同的 x_start')
conv_snaps = g_conv.run_with_snapshots(
    t_end=T_TRACK, x_start=conv_x_start, frame_times=weld_frame_times,
    snapshot_dtype=np.float32,
)
if len(snaps) != len(conv_snaps) or not np.array_equal(
        [s[0] for s in snaps], [s[0] for s in conv_snaps]):
    raise RuntimeError('传导/对流动画帧未对齐')
Lc, Wc, Dc = g_conv.pool_size()
print(f"对流增强 alpha_eff/alpha = {g_conv.k_pool_mult:.1f} "
      f"(dgamma/dT = {g_conv.convection.dgamma_dT:+.1e} N/(m K), 外向流)")
print(f"熔池 L×W×D = {Lc:.1f}×{Wc:.1f}×{Dc:.1f} mm (传导 {L:.1f}×{W:.1f}×{D:.1f}), "
      f"T_max = {g_conv.peak.max():.0f} K (传导 {g.peak.max():.0f} K)")
cache_mib = sum(a.nbytes for frames in (snaps, conv_snaps)
                for _, *arrays in frames for a in arrays) / 1024**2
print(f"PyVista 动画缓存: {len(snaps)} 帧 × 传导/对流 = {cache_mib:.0f} MiB")

## 5. 合成场景: 机械臂 + 动态焊接熔池

温度场从 FDM 域坐标映射进机器人基座系 (焊缝起点 `P0` ↔ 热源起点
`x_start`, 深度向下), 与逐帧机械臂同场渲染: 红色等温面为熔合区
(T ≥ Tm), 黄色半透明为 HAZ (T ≥ 1073 K), 板顶面为温度切片。

**瞬时熔池 g.T vs 熔合区历史 g.peak** — 红等值面/HAZ/顶面切片
(勾选对流时含 `g_conv` 蓝面) 整体切换: `g.peak` 是曾经熔过的包络
(长拖尾 + 鱼鳞纹), `T` 是当前帧枪尖处的**当下熔池**。Play/帧滑块
同步更新机械臂、实际 TCP 轨迹、熔池、HAZ、切片和标尺；模式切换
直接控制同一持久 pipeline 中的 actor，不移动相机。

**叠加对流修正熔合区** (§4b, 蓝色): 勾选后传导熔合区调为半透明,
可以看到对流修正池嵌在其内 — 更短、更光滑 (池内混合抹平了逐摆动
周期的"鱼鳞"棱脊)。此项同样原位切换并保持相机/对象位姿。

取景: 鼠标手势即时但不持久 (同 §2); **视图滑块** (平移 + 旋转 +
臂透明) 在默认取景 (对准工件 + 焊枪) 基础上做可复现的调整 — 平移 x
拉到 -400 mm 移到机器人基座; 默认视角下焊枪锥正压着熔池头部,
**臂透明拉到 100% 整臂移出场景**。复选框与视图滑块都持久化,
重跑后原样恢复。本格准备 live panel；§5b 完成 GIF 后统一显示
GIF / Composite PyVista / Seam PyVista 三页 tab。

两个 live PyVista 页都提供橙色 **TCP 拖动手柄**：直接左拖枪尖会暂停
播放，并用 Mink 逆运动学实时改变 UR5e 构型，同时保持拖动开始时的工具
姿态。该手动构型只覆盖当前帧；继续播放或移动帧滑块会恢复对应帧的记录构型。

In [ ]:
# FDM (x, y, z=深度) -> 机器人基座系 [mm]; 传导(xfine)/对流(fine)网格不同,
# 各自构造 StructuredGrid, 等值面位置是物理的, 可直接同场叠加。
# 每个网格带两个标量场: 'peak' = 熔合区历史 (曾经熔过的包络),
# 'inst' = t_end 末时刻的瞬时场 (当下的池)。两组热层都导出, 带透明度
# 指纹 (TAG_PEAK/TAG_INST), 场景内复选框客户端切换 —— 视角保持不变。
def make_grid(gg):
    ggx = (gg.x - cfg.run.goldak.x_start + P0[0])*1e3
    ggy = (gg.y + P0[1])*1e3
    ggz = (P0[2] - gg.z)*1e3
    X, Y, Z = np.meshgrid(ggx, ggy, ggz, indexing='ij')
    sg = pv.StructuredGrid(X, Y, Z)
    sg['peak'] = gg.peak.ravel(order='F')         # 单元(i,j,k) -> ravel('F')
    sg['inst'] = gg.T.ravel(order='F')
    return sg, ggx, ggz


# 旧 html fallback 的网格按需手工创建；live frontend 不做这份重复分配。
# grid, gx, gz = make_grid(g)
# grid_conv, _, _ = make_grid(g_conv)


def show_scene(show_conv, show_inst, px, py, pz, az, el, roll, arm_t):
    grid, gx, gz = make_grid(g)
    grid_conv, _, _ = make_grid(g_conv)
    p = pv.Plotter(notebook=True, window_size=(1000, 640))
    p.set_background("white")
    # 臂透明滑块: 焊枪锥在默认取景里正压着熔池头部, 调透明/拉满即可透视焊缝
    add_arm(p, q_at(T_TRACK), opacity=1.0 - arm_t/100.0)
    p.add_mesh(grid.outline(), color="#888888")
    # 两个字段的热层都加入, 按指纹分组; 初始可见性由注入脚本按 show_inst 设置
    for field, tag in (('peak', TAG_PEAK), ('inst', TAG_INST)):
        # 叠加对流池时传导熔合区调半透明, 以便看到嵌在其内的对流修正池
        p.add_mesh(grid.contour([float(g.Tm)], scalars=field),
                   color=(0.85, 0.1, 0.1),
                   opacity=(0.35 if show_conv else 0.9) + tag)
        if show_conv:
            p.add_mesh(grid_conv.contour([float(g_conv.Tm)], scalars=field),
                       color=(0.15, 0.35, 0.9), opacity=0.95 + tag)
        p.add_mesh(grid.contour([1073.0], scalars=field),
                   color=(1.0, 0.85, 0.2), opacity=0.25 + tag)
        top = grid.slice(normal='z', origin=(gx.mean(), 0.0, gz.max() - 1e-3))
        active = field == ('inst' if show_inst else 'peak')
        p.add_mesh(top, scalars=field, cmap='jet', opacity=0.8 + tag,
                   show_scalar_bar=active,   # 标尺只挂初始字段 (客户端切换后不更新)
                   scalar_bar_args={'title': f'T {field} [K]'})
    pts = tip*1e3 + [0, 0, 0.3]
    p.add_mesh(pv.lines_from_points(pts).tube(radius=0.6), color="#d81b1b")
    p.add_text("robot-executed weave pool"
               + (" + 10A convection (blue)" if show_conv else ""),
               font_size=11, color="black")
    add_mouse_hint(p)
    p.add_axes(xlabel="x [mm]", ylabel="y [mm]", zlabel="z [mm]")
    # 默认取景对准 工件 + 焊枪 (滚轮缩小可见机器人全身; §2 已有全身场景);
    # 视图滑块在此基础上平移/旋转 (如 平移 x=-400 mm 移到机器人基座)
    focal = np.array([gx.mean(), 0.0, gz.max()])
    p.camera.focal_point = tuple(focal)
    p.camera.position = tuple(focal + [210, -320, 230])
    p.camera.up = (0.0, 0.0, 1.0)
    apply_view(p, px, py, pz, az, el, roll)
    return html_view(p, pool_toggle=(POOL_LABEL, show_inst))


# Legacy static renderer kept above for reference; live widget starts here.
close_output_tabs(globals())
close_app(globals(), 'composite_pv')
close_app(globals(), 'seam_pv')
composite_layers = (
    LayerSpec('robot', 'UR5e robot'),
    LayerSpec('domain', 'Goldak grid outline'),
    LayerSpec('executed_path', 'Executed path to current time'),
    LayerSpec('thermal', 'Conductive melt / HAZ / top slice'),
    LayerSpec('instant_mode', 'Instantaneous field (off: peak history)', False),
    LayerSpec('convection', '10A convection overlay', False),
    LayerSpec('axes', 'Local axes'),
)
composite_scene = CompositePVScene(
    pv_context, g_gif, g_conv, snaps, conv_snaps, x_start=x_start,
    layers=composite_layers,
    current_frame=initial_frame(pv_store, 'scene5', weld_frame_times, 0.0),
)
composite_pv = PyVistaWidgetApp(
    composite_scene, layers=composite_layers, store=pv_store,
    scene_name='scene5', title='UR5e + robot-executed weld pool — PyVista',
    height=710, frame_ms=125, lim=500.0, step=20.0,
    enable_tip_ik=True,
)
print('Composite PyVista 已准备；运行 §5b 后在统一 tab 中显示。')

## 5b. 动画: 焊缝成形 GIF + PyVista 实时回放

复用 §4b 的 41 帧传导快照，写出 `results/robot6_weave_seam.gif`
(8 fps; Pillow 写出, 不新增 imageio 依赖):
机械臂沿摆动轨迹移动枪尖, 枪尖处**亮红**为瞬时熔池 (T ≥ Tm),
身后**暗红**为已凝固焊缝 (peak 包络逐帧生长, 摆动扇贝纹清晰可见),
顶面为 ≥400 K 热晕 (inferno 切片, 冷板不遮挡)。取景对准焊缝中段,
腕部/焊枪的摆动与行进清晰可辨。

`SeamPVScene` 持有持久 structured grid 和 topology-changing filters；
每帧只更新 NumPy/VTK 标量数组。相同 PyVista render window 先抓取 GIF，
随后作为 live widget 播放。最终输出一个三页 tab：GIF animation（默认）、
Composite PyVista 和 Seam PyVista；离开 live 页会暂停播放。两个 live 页
均可拖动橙色 TCP 手柄实时调整机械臂；拖动会暂停播放，继续播放或改变帧
滑块时恢复该帧的仿真构型。

In [ ]:
from PIL import Image as PILImage

GIF_PATH = Path("../results/robot6_weave_seam.gif")
GIF_PATH.parent.mkdir(exist_ok=True)
if not snaps:
    raise RuntimeError('快照为空；请先运行 §4b')
if globals().get('composite_pv') is None or composite_pv.closed:
    raise RuntimeError('复合 PyVista 场景尚未准备；请先运行 §5')

close_output_tabs(globals())
close_app(globals(), 'seam_pv')
seam_layers = (
    LayerSpec('robot', 'UR5e robot'),
    LayerSpec('workpiece', 'Workpiece'),
    LayerSpec('seam', 'Seam centreline'),
    LayerSpec('executed_path', 'Executed path to current time'),
    LayerSpec('history', 'Solidified / peak envelope'),
    LayerSpec('pool', 'Instantaneous molten pool'),
    LayerSpec('halo', 'Top-surface thermal halo'),
    LayerSpec('annotation', 'Time / colour legend'),
    LayerSpec('axes', 'Local axes'),
)
seam_scene = SeamPVScene(
    pv_context, g_gif, snaps, x_start=x_start, layers=seam_layers,
)

# 同一个 PyVista render window 既生成 GIF，也交给 live widget 播放。
frames = []
for index in range(seam_scene.frame_count):
    seam_scene.set_frame(index, render=False)
    frames.append(PILImage.fromarray(seam_scene.capture_rgb()))
seam_scene.set_frame(0, render=False)
frames[0].save(
    GIF_PATH, save_all=True, append_images=frames[1:], loop=0,
    duration=[125]*(len(frames) - 1) + [1500], optimize=True,
)
gif_summary = (f'{GIF_PATH} · {len(frames)} 帧 · '
               f'{GIF_PATH.stat().st_size/1024:.0f} kB · PyVista server')

seam_pv = PyVistaWidgetApp(
    seam_scene, layers=seam_layers, store=pv_store, scene_name='scene5b',
    title='Robot weave seam formation — PyVista', height=650,
    frame_ms=125, lim=180.0, step=10.0,
    enable_tip_ik=True,
)
weld_output_tabs = build_output_tabs(
    globals(), GIF_PATH, gif_summary, composite_pv, seam_pv,
)
display(weld_output_tabs)


## 相关

- **可复用工具箱**: `utils/robot6_pv_widget.py` 提供 PyVista server
  widget 的持久场景、动态 VTK pipeline、摄影棚环境、世界坐标轴、
  图层/播放控件、对象选择变换与 tab 生命周期；`WidgetStore` 负责取景/
  图层/时间持久化。旧 `pv_inline.py` 仍供只读 html 场景使用。§5b 的
  GIF 另有独立脚本 `uv run python notebooks/utils/make_seam_gif.py`。
- **定量部署分析** (摆频扫描 0.5–4 Hz 幅值保真度、梯形波超调、理想 vs
  执行熔池对比): `robot_deployment_scenarios.ipynb`。
- **熔池对流**: 耦合路线评估见 `docs/melt_convection_assessment.md`;
  三个 Marangoni 降阶模型 (10A/10B/10C) 的独立演示见
  `marangoni_comparison.ipynb`。
- **五个典型工况 + 交互式体渲染**: `welding_scenarios_interactive_demo.ipynb`;
  PyVista 内联渲染的后端/裁剪切面技巧: `pyvista_interactive_demo.ipynb`。
- **CLI**: 机械臂演示 `uv run welding-sim-vi` (模块 7b, m7b_robot6_vi.png);
  热场 `uv run welding-sim-3d process=db_median weave=triangle solver=fine`
  (加 `convection=effective` 得 §4b 的对流增强场)。
- **玩法**: 改 §1 的 `weave`(`compose_cfg("sim_3d", "weave=pattern1")`) 看
  梯形摆的超调轨迹; 改 `P0` / `R_REF` 摆放焊缝与焊枪姿态 (立焊/横焊,
  注意热场为传导 + 有效导热率近似, 无重力/位置依赖, 焊接位置只影响
  机器人侧); 换回默认球腕臂 (`compose_cfg("sim_vi").robot6`) 或改
  `conf/model/robot6_ur5e.yaml` 自定义机型 — `SixDofArm` 接受任意标准
  DH 表 (`dh_d/dh_a/dh_alpha_deg`)。